# 01 - Data versioning with DVC

This replaces the workshop's SageMaker Feature Store entirely.

### Why

Feature Store gave two things: a managed home for feature data, and online
lookup at inference time. For *training* reproducibility it gave surprisingly
little -- the old notebook 02 queried "the most recent record per player", so
re-running it after any ingest produced a different dataset with no way to
recover the old one. The dataset was whatever the store happened to contain.

DVC inverts that. Data lives in S3 content-addressed by hash; git holds small
pointer files. A commit therefore pins an exact dataset, and `dvc checkout`
materialises it. You get:

- **Reproducibility.** `git checkout <sha> && dvc checkout` is byte-identical, always.
- **Diffability.** `dvc diff` / `dvc status` tell you what data changed between commits.
- **No idle cost.** No online store to keep warm.

What you give up: online single-record feature lookup. This model scores batches
of players, so that was never used. (If you later need it, the clean split is
DVC for the training path and Feature Store purely as a serving cache.)

### Layout

```
data/raw/player_churn.csv        <- dvc add       (pointer: player_churn.csv.dvc)
data/interim/features.parquet    <- dvc stage out (hashed in dvc.lock)
data/processed/                  <- dvc stage out (hashed in dvc.lock)
```

Only the raw file needs `dvc add`. Stage outputs are tracked by `dvc.yaml`.

In [1]:
import os
import subprocess
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
os.environ["CHURN_PROJECT_ROOT"] = str(PROJECT_ROOT)

# Load the values notebook 00 wrote.
for line in (PROJECT_ROOT / ".env").read_text().strip().splitlines():
    key, _, value = line.partition("=")
    os.environ[key] = value

from src.churn.config import Settings, load_params

params = load_params()
settings = Settings.load(params, require_role=False)
print("bucket    :", settings.bucket)
print("dvc remote:", settings.dvc_remote_url)

bucket    : sagemaker-us-east-1-063676201336
dvc remote: s3://sagemaker-us-east-1-063676201336/player-churn/dvc-store


In [2]:
def run(cmd, check=True):
    """Run a shell command from the project root and echo its output."""
    print("$", cmd)
    proc = subprocess.run(cmd, shell=True, cwd=PROJECT_ROOT, capture_output=True, text=True)
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print(proc.stderr.rstrip())
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed ({proc.returncode}): {cmd}")
    return proc

## 1. Initialise DVC

`--subdir` matters. This project is a folder inside a larger git repo, and plain
`dvc init` insists on sitting at the git root. `--subdir` puts `.dvc/` here
instead, so `churn-mlops/` is self-contained and the workshop notebooks in the
parent folder are untouched.

In [3]:
if not (PROJECT_ROOT / ".dvc").exists():
    run("dvc init --subdir")
else:
    print(".dvc already exists, skipping init")

run("dvc config core.autostage true")   # `dvc add` git-stages its own pointer files

$ dvc init --subdir
Initialized DVC repository.

You can now commit the changes to git.

+---------------------------------------------------------------------+
|                                                                     |
|        DVC has enabled anonymous aggregate usage analytics.         |
|     Read the analytics documentation (and how to opt-out) here:     |
|             <https://dvc.org/doc/user-guide/analytics>              |
|                                                                     |
+---------------------------------------------------------------------+

What's next?
------------
- Check out the documentation: <https://dvc.org/doc>
- Get help and share ideas: <https://dvc.org/chat>
- Star us on GitHub: <https://github.com/treeverse/dvc>
$ dvc config core.autostage true


CompletedProcess(args='dvc config core.autostage true', returncode=0, stdout='', stderr='')

## 2. Point DVC at S3

The remote lives under its own prefix (`aws.dvc_remote_prefix` in `params.yaml`)
so DVC's content-addressed blobs never mingle with the plain-CSV copies that
training jobs read.

The Studio execution role already has S3 access, so there is no credential to
configure.

In [4]:
run(f"dvc remote add --default --force s3remote {settings.dvc_remote_url}")
run(f"dvc remote modify s3remote region {settings.region}")
run("dvc remote list")

$ dvc remote add --default --force s3remote s3://sagemaker-us-east-1-063676201336/player-churn/dvc-store
Setting 's3remote' as a default remote.
$ dvc remote modify s3remote region us-east-1
$ dvc remote list
s3remote        s3://sagemaker-us-east-1-063676201336/player-churn/dvc-store    
(default)


CompletedProcess(args='dvc remote list', returncode=0, stdout='s3remote        s3://sagemaker-us-east-1-063676201336/player-churn/dvc-store    \n(default)\n', stderr='')

## 3. Track the raw dataset

The CSV came with the parent workshop repo. We copy it in and hand it to DVC;
git gets a ~100-byte pointer instead of the file.

In [5]:
raw = PROJECT_ROOT / "data" / "raw" / "player_churn.csv"
if not raw.exists():
    source = PROJECT_ROOT.parent / "data" / "player_churn.csv"
    raw.parent.mkdir(parents=True, exist_ok=True)
    raw.write_bytes(source.read_bytes())
    print(f"copied {source} -> {raw}")

print(f"{raw.name}: {raw.stat().st_size / 1024:.0f} KiB")

player_churn.csv: 600 KiB


In [6]:
run("dvc add data/raw/player_churn.csv")

$ dvc add data/raw/player_churn.csv
⠋ Checking graph



CompletedProcess(args='dvc add data/raw/player_churn.csv', returncode=0, stdout='', stderr='\x1b[?25l⠋ Checking graph\n\x1b[?25h\n')

In [7]:
# The pointer file is what git stores. This is the whole trick.
print((PROJECT_ROOT / "data/raw/player_churn.csv.dvc").read_text())
print("--- and DVC wrote the data path into .gitignore ---")
print((PROJECT_ROOT / "data/raw/.gitignore").read_text())

outs:
- md5: 2771ef23f604095528a26e93e76f9fc0
  size: 614041
  hash: md5
  path: player_churn.csv

--- and DVC wrote the data path into .gitignore ---
# `dvc add data/raw/player_churn.csv` takes ownership of this file and manages
# the entry below. The companion player_churn.csv.dvc pointer IS committed.
/player_churn.csv



## 4. Push the data

`dvc push` uploads blobs the remote does not already have. Re-running it after
an unchanged file is a no-op -- it compares hashes, not timestamps.

In [8]:
run("dvc push")
run("dvc status --cloud", check=False)   # expect "Cache and remote are in sync"

$ dvc push
1 file pushed
$ dvc status --cloud
Cache and remote 's3remote' are in sync.
You can also use `dvc commit features` to associate existing 'data/interim/features.parquet' with stage: 'features'.
You can also use `dvc commit split` to associate existing 'data/processed' with stage: 'split'.


CompletedProcess(args='dvc status --cloud', returncode=0, stdout="Cache and remote 's3remote' are in sync.\n", stderr="WARNING: Output 'data/interim/features.parquet'(stage: 'features') is missing version info. Cache for it will not be collected. Use `dvc repro` to get your pipeline up to date.\nYou can also use `dvc commit features` to associate existing 'data/interim/features.parquet' with stage: 'features'.\nWARNING: Output 'data/processed'(stage: 'split') is missing version info. Cache for it will not be collected. Use `dvc repro` to get your pipeline up to date.\nYou can also use `dvc commit split` to associate existing 'data/processed' with stage: 'split'.\n")

## 5. Commit the pointers

Two commits' worth of discipline, every time data changes:

```sh
dvc push       # data to S3
git commit     # pointers to GitHub
```

Push the data **before** the commit. A commit whose pointers reference blobs
nobody can fetch is worse than no commit.

In [9]:
run("git status --short -- churn-mlops", check=False)

$ git status --short -- churn-mlops


CompletedProcess(args='git status --short -- churn-mlops', returncode=0, stdout='', stderr='')

In [10]:
# Run this yourself when you are happy with the state:
print(
    """
cd {root}
git add .dvc .dvcignore dvc.yaml params.yaml pyproject.toml requirements*.txt \\
        src tests notebooks scripts data/raw/.gitignore data/raw/player_churn.csv.dvc
git commit -m "churn-mlops: DVC-tracked dataset and pipeline skeleton"
""".format(root=PROJECT_ROOT)
)


cd /Users/ritabanchowdhury/Desktop/mlops-sagemaker-mlflow/churn-mlops
git add .dvc .dvcignore dvc.yaml params.yaml pyproject.toml requirements*.txt \
        src tests notebooks scripts data/raw/.gitignore data/raw/player_churn.csv.dvc
git commit -m "churn-mlops: DVC-tracked dataset and pipeline skeleton"



## 6. What versioning actually buys you

The point is recovery. Say someone replaces the extract:

```sh
# a new extract lands
cp /path/to/new_extract.csv data/raw/player_churn.csv
dvc add data/raw/player_churn.csv      # pointer hash changes
dvc push && git commit -am "new extract"

# the model regresses; go back
git checkout HEAD~1 -- data/raw/player_churn.csv.dvc
dvc checkout                           # the OLD bytes are restored
```

`dvc status` compares your working tree against the pointers; `dvc diff` compares
two commits.

In [11]:
run("dvc status", check=False)
run("dvc data status", check=False)

$ dvc status
features:
	changed deps:
		modified:           data/raw/player_churn.csv
		modified:           src/churn/features.py
		new:                params.yaml
	changed outs:
		modified:           data/interim/features.parquet
split:
	changed deps:
		modified:           data/interim/features.parquet
		modified:           src/churn/split.py
		new:                params.yaml
	changed outs:
		modified:           data/processed
		modified:           reports/split_summary.json
$ dvc data status
DVC committed changes:
  (git commit the corresponding dvc files to update the repo)
        added: data/raw/player_churn.csv

DVC uncommitted changes:
  (use "dvc commit <file>..." to track changes)
  (use "dvc checkout <file>..." to discard changes)
        added: data/interim/features.parquet
        added: data/processed/
(there are other changes not tracked by dvc, use "git status" to see)


CompletedProcess(args='dvc data status', returncode=0, stdout='DVC committed changes:\n  (git commit the corresponding dvc files to update the repo)\n        added: data/raw/player_churn.csv\n\nDVC uncommitted changes:\n  (use "dvc commit <file>..." to track changes)\n  (use "dvc checkout <file>..." to discard changes)\n        added: data/interim/features.parquet\n        added: data/processed/\n(there are other changes not tracked by dvc, use "git status" to see)\n', stderr="WARNING: Commit '694fa80' does not contain a DVC repo\n")

---
## Next

**`02-pipeline.ipynb`** -- define the raw → features → splits DAG and run it.

### What changed vs. the original workshop
Old notebook 01 ended with `feature_group.ingest(...)` and a feature-group name
you pasted into notebook 02. There is no name to paste now: the dataset is
addressed by git commit. Old notebook 01 also oversampled the minority class
*before* the split -- notebook 02 explains why that moved.